# Execution Pipeline
Automated end-to-end execution on Google Colab:
1. Direct one-line public git clone (zero PAT / zero authentication needed).
2. Google Drive / high-speed CDN extraction.
3. Multi-stage Entity Resolution pipeline execution.
4. Validation & Packaging directly into `PERCEPTRON_submission.zip`.

--- 
## Step 1: Clone Repository (No PAT Needed!)
Direct public clone without any tokens or password prompts.

In [ ]:
!rm -rf /content/sys-eval-core-x26
!git clone https://github.com/SabarishR08/sys-eval-core-x26.git /content/sys-eval-core-x26
%cd /content/sys-eval-core-x26
!pip install -r code/business_entity_resolution/requirements.txt

--- 
## Step 2: Mount Google Drive (Optional)
Mount Google Drive if you uploaded `amazon_ml_student_resource.zip` or `6ab10eb3b23ba_student_resource.zip` to Drive.

In [ ]:
from google.colab import drive
import os, sys, shutil, zipfile
from pathlib import Path

try:
    drive.mount('/content/drive')
except Exception:
    print('Google Drive mount skipped.')

--- 
## Step 3: Extract Dataset (from Google Drive OR Direct CDN Download)
If not found in Drive, it downloads directly from the official Unstop CDN at full speed.

In [ ]:
zip_names = ['amazon_ml_student_resource.zip', '6ab10eb3b23ba_student_resource.zip']
zip_path = None

# 1. Check Google Drive
if os.path.exists('/content/drive/MyDrive'):
    for zn in zip_names:
        found = list(Path('/content/drive/MyDrive').rglob(zn))
        if found:
            zip_path = str(found[0])
            print(f'Found dataset zip in Google Drive: {zip_path}')
            break

# 2. If not found in Drive, download directly from official CDN
if not zip_path:
    cdn_url = 'https://cdn.unstop.com/files/6ab10eb3b23ba_student_resource.zip'
    zip_path = '/content/amazon_ml_student_resource.zip'
    print(f'Zip not found in Google Drive. Downloading directly from Unstop CDN: {cdn_url}...')
    !wget -q --show-progress -O /content/amazon_ml_student_resource.zip 'https://cdn.unstop.com/files/6ab10eb3b23ba_student_resource.zip'

print(f'Extracting {zip_path} directly onto fast NVMe disk (/content/sys-eval-core-x26)...')
with zipfile.ZipFile(zip_path, 'r') as z:
    for item in z.infolist():
        if item.filename.startswith('__MACOSX') or item.filename.endswith('.DS_Store'):
            continue
        rel = item.filename
        if rel.startswith('student_resource/'):
            rel = rel[len('student_resource/'):]
        if not rel:
            continue
        target = os.path.join(os.getcwd(), rel)
        if item.is_dir():
            os.makedirs(target, exist_ok=True)
        else:
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with z.open(item) as src, open(target, 'wb') as dst:
                dst.write(src.read())
print('Extraction complete! All train and test TSV files ready.')

--- 
## Step 4: Verify Extracted TSV Files

In [ ]:
!ls -lh dataset/train
!ls -lh dataset/test

--- 
## Step 5: Execute Complete Entity Resolution Pipeline

In [ ]:
!python code/business_entity_resolution/src/pipeline.py \
    --train-dir dataset/train \
    --test-dir dataset/test \
    --output-dir output \
    --threshold 0.68

--- 
## Step 6: Validate and Package Final Submission
Validates with the official competition checker and saves `PERCEPTRON_submission.zip` to Google Drive.

In [ ]:
!python package_submission.py --team-name PERCEPTRON

# Copy to Google Drive if mounted
if os.path.exists('/content/drive/MyDrive'):
    drive_output_path = '/content/drive/MyDrive/PERCEPTRON_submission.zip'
    shutil.copy('PERCEPTRON_submission.zip', drive_output_path)
    print(f'Successfully copied final submission zip to: {drive_output_path}')